# Feedback System

## Problem Statement

Recommendation systems become more accurate over time by learning from users' responses to recommended products.

This learning process is known as a **Feedback System**.

A feedback system records how users react to recommendations, such as whether they viewed, clicked, added to cart, or purchased the recommended products.

The collected feedback can then be used to improve future recommendations and personalize the user experience.

---

## Why is Feedback Important?

A recommendation model generates predictions based on historical user behavior.

However, user preferences change over time.

By continuously collecting user feedback, the recommendation system can:

- Learn changing user preferences
- Improve recommendation quality
- Reduce irrelevant recommendations
- Increase user engagement
- Improve customer satisfaction

---

## Dataset Limitation

The RetailRocket dataset contains only historical user interactions.

It does **not** contain user interactions that happened **after** recommendations were generated.

Therefore, real production feedback cannot be obtained from this dataset.

To demonstrate an industry-level recommendation pipeline, this notebook simulates the feedback collection process that would normally occur in a production environment after recommendations are shown to users.

The generated feedback dataset will later be used by the Personalization Engine.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime

pd.set_option(
    "display.max_columns",
    None
)

In [ ]:
recommendations_df = pd.read_csv(
    "../data/features/recommendations.csv"
)

customer_segments = pd.read_csv(
    "../data/features/customer_segments.csv"
)

In [ ]:
print("Recommendations Shape :", recommendations_df.shape)

print("Customer Segments Shape :", customer_segments.shape)

In [ ]:
recommendations_df.head()

In [ ]:
customer_segments.head()

Before generating feedback, let us understand what each recommendation represents.

Each row indicates:

> "This product has been recommended to this user."

At this stage, we do **not** know whether the user actually interacted with the recommended product.

The purpose of the Feedback System is to record those future user interactions.

In [ ]:
recommendations_df.info()

In [ ]:
recommendations_df.describe()

Since the RetailRocket dataset does not contain post-recommendation user interactions, we create a copy of the recommendation dataset.

This copied dataset will be enriched with simulated feedback information in the next steps.

The original recommendation dataset remains unchanged.

In [ ]:
feedback_df = recommendations_df.copy()

In [ ]:
feedback_df.head()

### Understanding Simulated Feedback

In a real production recommendation system:

1. The Recommendation Engine suggests products.
2. Users interact with those recommendations.
3. Those interactions are stored as feedback.
4. The Personalization Engine learns from that feedback.

Since RetailRocket is a historical dataset, these future interactions are unavailable.

Therefore, we simulate realistic user feedback using the recommendation confidence score.

Products with higher recommendation scores have a higher probability of receiving positive feedback than products with lower recommendation scores.

This creates a realistic demonstration of an online recommendation pipeline while acknowledging the dataset's limitations.

### Simulate User Feedback

Since the RetailRocket dataset does not contain explicit user ratings for recommended products, we simulate feedback based on the recommendation score.

The simulation follows these assumptions:

- High recommendation score → Higher probability of positive feedback.
- Medium recommendation score → Moderate probability.
- Low recommendation score → Higher probability of negative feedback.

Possible feedback values:

- Like
- Neutral
- Dislike

This simulated feedback enables us to demonstrate a complete recommendation feedback loop for educational and portfolio purposes.

In [ ]:
import numpy as np

np.random.seed(42)

feedback = []

for score in recommendations_df["recommendation_score"]:

    if score >= 0.80:
        feedback.append(
            np.random.choice(
                ["Like", "Neutral"],
                p=[0.85, 0.15]
            )
        )

    elif score >= 0.60:
        feedback.append(
            np.random.choice(
                ["Like", "Neutral", "Dislike"],
                p=[0.55, 0.30, 0.15]
            )
        )

    elif score >= 0.40:
        feedback.append(
            np.random.choice(
                ["Like", "Neutral", "Dislike"],
                p=[0.30, 0.40, 0.30]
            )
        )

    else:
        feedback.append(
            np.random.choice(
                ["Neutral", "Dislike"],
                p=[0.20, 0.80]
            )
        )

recommendations_df["feedback"] = feedback

recommendations_df.head()

In [ ]:
recommendations_df["feedback"].value_counts()

In [ ]:
import matplotlib.pyplot as plt

feedback_counts = recommendations_df["feedback"].value_counts()

plt.figure(figsize=(7,5))

feedback_counts.plot(
    kind="bar",
    color=["green","orange","red"]
)

plt.title("Distribution of Simulated Feedback")

plt.xlabel("Feedback")

plt.ylabel("Count")

plt.xticks(rotation=0)

plt.show()

### Convert Feedback into Numerical Scores

Machine learning models require numerical inputs rather than text labels.

Therefore, the simulated feedback is converted into numerical values.

Mapping used:

Like      → 1

Neutral   → 0

Dislike   → -1

These numerical scores will later be used to adjust recommendation scores during personalization

In [ ]:
feedback_mapping = {
    "Like": 1,
    "Neutral": 0,
    "Dislike": -1
}

recommendations_df["feedback_score"] = (
    recommendations_df["feedback"]
    .map(feedback_mapping)
)

recommendations_df.head()

In [ ]:
recommendations_df[
    ["feedback","feedback_score"]
].head(15)

In [ ]:
recommendations_df["feedback_score"].value_counts()

### Save Feedback Dataset

The recommendation dataset is now enriched with simulated user feedback.

Saving this intermediate dataset provides a reusable input for the Personalization Engine notebook without requiring feedback simulation again.

In [ ]:
recommendations_df.to_csv(
    "../data/features/recommendations_with_feedback.csv",
    index=False
)

print("Feedback dataset saved successfully.")

### Understanding Feedback-Based Personalization

Collecting user feedback is only valuable if the recommendation system learns from it.

In production recommendation systems, user interactions continuously influence future recommendations.

For example:

- Products receiving positive feedback become more likely to be recommended again.
- Products receiving negative feedback gradually lose importance.
- Neutral feedback keeps the recommendation score largely unchanged.

In this project, we simulate this behavior by adjusting each recommendation score according to the simulated user feedback.

This demonstrates how recommendation systems evolve over time based on user preferences.

### Adjust Recommendation Scores

Each recommendation score is updated using the numerical feedback score.

Adjustment strategy:

- Like (+1): Increase recommendation score
- Neutral (0): Keep score unchanged
- Dislike (-1): Decrease recommendation score

The adjustment is intentionally small to imitate gradual learning rather than sudden changes in user preferences

In [ ]:
# Adjustment factor
learning_rate = 0.10

'''This calculates a new recommendation score by combining:

Original recommendation score (generated by collaborative filtering)
Feedback score (derived from user interactions)
Learning rate (controls the impact of feedback)'''


recommendations_df["updated_score"] = (
    recommendations_df["recommendation_score"]
    + learning_rate * recommendations_df["feedback_score"]
)

# Keep scores within valid range
recommendations_df["updated_score"] = recommendations_df["updated_score"].clip(0, 1)

recommendations_df.head()

Compare Original and Updated Scores

After incorporating user feedback, we compare the original recommendation scores with the updated scores.

This comparison illustrates how positive and negative feedback influence future recommendations.

In [ ]:
recommendations_df[
    [
        "visitorid",
        "itemid",
        "recommendation_score",
        "feedback",
        "updated_score"
    ]
].head(20)

In [ ]:
#Measure Score Changes
#Positive values indicate improved recommendations, while negative values indicate reduced recommendation confidence.
recommendations_df["score_change"] = (
    recommendations_df["updated_score"]
    - recommendations_df["recommendation_score"]
)

recommendations_df.head()

In [ ]:
recommendations_df["score_change"].describe()

In [ ]:
#Visualize Score Changes
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.hist(
    recommendations_df["score_change"],
    bins=20
)

plt.title("Distribution of Recommendation Score Changes")

plt.xlabel("Score Change")

plt.ylabel("Frequency")

plt.show()

### Generate Updated Rankings

After updating recommendation scores, products are re-ranked for each user.

Products with higher updated scores move upward in the recommendation list, while products with lower scores move downward.

This process represents the learning behavior of a personalized recommendation system.

In [ ]:
recommendations_df = recommendations_df.sort_values(
    ["visitorid", "updated_score"],
    ascending=[True, False]
)

In [ ]:
#ranking products for each user 
recommendations_df["updated_rank"] = (
    recommendations_df
    .groupby("visitorid")
    .cumcount()  #counts how many items recommended for each user 
    + 1
)

recommendations_df.head(20)

In [ ]:
recommendations_df[
    [
        "visitorid",
        "itemid",
        "updated_rank",
        "feedback",
        "recommendation_score",
        "updated_score"
    ]
].head(30)

### Save Personalized Recommendations

The updated recommendations now reflect simulated user preferences.

Saving this dataset allows the Personalization Engine, API, and deployment pipeline to use feedback-adjusted recommendations without repeating the feedback learning process.

In [ ]:
recommendations_df.to_csv(
    "../data/features/personalized_recommendations.csv",
    index=False
)

print("Personalized recommendations saved successfully.")